In [28]:
import openai
import langchain_core
from langchain_openai.chat_models import ChatOpenAI
from langchain.tools import Tool
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

from langchain.memory.buffer import ConversationBufferMemory
from langchain.memory.chat_memory import BaseChatMemory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, AIMessage
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder)
#from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.runnables import (
    RunnableLambda,
    ConfigurableFieldSpec,
    RunnablePassthrough,
)
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.runnables.history import RunnableWithMessageHistory
from transformers import (AutoModelForCausalLM, AutoTokenizer, pipeline)
#from langchain.llms import HuggingFacePipeline
import uuid
import os
import torch
from langchain_ollama import ChatOllama
import ollama

In [ ]:
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"

In [30]:
kuhn_poker_rules = """You are playing a single round of a Poker game as player 1 against an opponent (player 2). Here are the game rules:
Each player antes 1 chip. Each player is dealt a unique card from a deck consisting of only a Jack (J), a Queen (Q), and a King (K). The third card not dealt is left unseen.
Player 1 can check or bet 1 chip.
If player 1 checks, player 2 can check or bet 1 chip.
  > If player 2 checks, a showdown occurs.
  > If player 2 bets, player 1 can fold or call.
      > If player 1 folds, player 2 takes the entire pot.
      > If player 1 calls, a showdown occurs.
If player 1 bets, player 2 can fold or call.
  > If player 2 folds, player one takes the entire pot.
  > If player 2 calls, a showdown occurs."""

inform_llm_of_opponent_strategy = "Your oppoent employs the follwoing strategy: "

nash = "with a K, always bet or call; with a Q, always check after you check, call with a probability of 1/3 after you bet; with a J, always fold after you bet, and bet with a probability of 1/3 after you check."
copy_player1 = "always call when you bet, and always check when you check."
random = "choose an action at random with equal probability."


def generate_prompt(llm_card,
                    rules = kuhn_poker_rules,
                    inform_llm_of_opponent_strategy="",
                    strategy_description="",
                    game_start="The game starts now.",
                    past_moves="",
                    question="What is your next move?"):
    
    prompt = rules + "\n" + f"{inform_llm_of_opponent_strategy} {strategy_description} {game_start} Your card is {llm_card}. {past_moves} {question}"    
    return prompt

print(generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash,
                      game_start="The game has started.",
                      past_moves="You checked, and your opponent bet.",
                      question="Now, what is your next move?"))

You are playing a single round of a Poker game as player 1 against an opponent (player 2). Here are the game rules:
Each player antes 1 chip. Each player is dealt a unique card from a deck consisting of only a Jack (J), a Queen (Q), and a King (K). The third card not dealt is left unseen.
Player 1 can check or bet 1 chip.
If player 1 checks, player 2 can check or bet 1 chip.
  > If player 2 checks, a showdown occurs.
  > If player 2 bets, player 1 can fold or call.
      > If player 1 folds, player 2 takes the entire pot.
      > If player 1 calls, a showdown occurs.
If player 1 bets, player 2 can fold or call.
  > If player 2 folds, player one takes the entire pot.
  > If player 2 calls, a showdown occurs.
Your oppoent employs the follwoing strategy:  with a K, always bet or call; with a Q, always check after you check, call with a probability of 1/3 after you bet; with a J, always fold after you bet, and bet with a probability of 1/3 after you check. The game has started. Your card i

In [31]:
client = openai.OpenAI()
def get_llm_response(model, prompt, method="chat completions", effort="medium"):
    if method == "chat completions":
        response = client.chat.completions.create(
                model=model,
                reasoning_effort=effort,
                messages=[
                    {
                    "role": "user",
                    "content": prompt
                    }
                    ],
                    
        )
        print(f"{model} with {method} says...")
        print(response.choices[0].message.content, "\n")
    
    elif method == "responses":
        response = client.responses.create(
                   model=model,
                   reasoning={"effort": effort,
                              "summary": "auto"},
                   input=[
                           {
                            "role": "user", 
                            "content": prompt
                            }
                         ]
        ) 
        print(f"{model} with {method} says...")
        print(response.output[1].content[0].text, "\n")
    return response


In [ ]:
prompt = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash)
print(prompt, "\n")
res_o1_chat_completions, res_o1_responses = get_llm_response("o1", prompt, method="chat completions"), get_llm_response("o1", prompt, method="responses")
print("\n\nReasoning Summary\n", res_o1_responses.output[0].summary)

In [ ]:
prompt_ = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash,
                      game_start="The game has started.",
                      past_moves="You checked, and your opponent bet.",
                      question="Now, what is your next move?")
print(prompt_, "\n")
res_o1_chat_completions_, res_o1_responses_ = get_llm_response("o1", prompt_, method="chat completions"), get_llm_response("o1", prompt_, method="responses")
print("\n\nReasoning Summary\n", res_o1_responses_.output[0].summary)

In [ ]:
prompt = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash)
print(prompt, "\n")
res_o3_chat_completions, res_o3_responses = get_llm_response("o3", prompt, method="chat completions"), get_llm_response("o3", prompt, method="responses")
print("\n\nReasoning Summary\n", res_o3_responses.output[0].summary)

In [ ]:
prompt_ = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash,
                      game_start="The game has started.",
                      past_moves="You checked, and your opponent bet.",
                      question="Now, what is your next move?")
print(prompt_, "\n")
res_o3_chat_completions_, res_o3_responses_ = get_llm_response("o3", prompt_, method="chat completions"), get_llm_response("o3", prompt_, method="responses")
print("\n\nReasoning Summary\n", res_o3_responses_.output[0].summary)

In [ ]:
prompt = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash)
print(prompt, "\n")
res_o3mini_chat_completions, res_o3mini_responses = get_llm_response("o3-mini", prompt, method="chat completions"), get_llm_response("o3-mini", prompt, method="responses")
print("\n\nReasoning Summary\n", res_o3mini_responses.output[0].summary)

In [ ]:
prompt_ = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash,
                      game_start="The game has started.",
                      past_moves="You checked, and your opponent bet.",
                      question="Now, what is your next move?")
print(prompt_, "\n")
res_o3mini_chat_completions_, res_o3mini_responses_ = get_llm_response("o3-mini", prompt_, method="chat completions"), get_llm_response("o3-mini", prompt_, method="responses")
print("\n\nReasoning Summary\n", res_o3mini_responses_.output[0].summary)

In [ ]:
prompt = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash)
print(prompt, "\n")
res_o4mini_chat_completions, res_o4mini_responses = get_llm_response("o4-mini", prompt, method="chat completions"), get_llm_response("o4-mini", prompt, method="responses")
print("\n\nReasoning Summary\n", res_o3mini_responses.output[0].summary)

In [ ]:
prompt_ = generate_prompt("Q",
                      inform_llm_of_opponent_strategy=inform_llm_of_opponent_strategy,
                      strategy_description=nash,
                      game_start="The game has started.",
                      past_moves="You checked, and your opponent bet.",
                      question="Now, what is your next move?")
print(prompt_, "\n")
res_o4mini_chat_completions_, res_o4mini_responses_ = get_llm_response("o4-mini", prompt_, method="chat completions"), get_llm_response("o4-mini", prompt_, method="responses")
print("\n\nReasoning Summary\n", res_o4mini_responses_.output[0].summary)